In [1]:
import lightgbm as lgb
import pandas as pd
import optuna
import warnings
import json

from i import input_dir, model_dir
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")

train = pd.read_csv(input_dir + "train.csv", index_col="id")
test = pd.read_csv(input_dir + "test.csv", index_col="id")
origin = pd.read_csv(input_dir + "diabetes_dataset.csv")

for col in train.select_dtypes(include="object").columns:
    train[col] = train[col].astype("category")
    test[col] = test[col].astype("category")
for col in origin.select_dtypes(include="object").columns:
    origin[col] = origin[col].astype("category")

Target_Col = "diagnosed_diabetes"

X = train.iloc[:, :-1]
y = train[Target_Col]
X_test = test

X_orig = origin[X.columns]
y_orig = origin[Target_Col]

with open(model_dir + "lgbm_base_params.json") as f:
    base_params = json.load(f)

c:\Users\Blanc\AppData\Local\pypoetry\Cache\virtualenvs\playground-lwZmZsxv-py3.12\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Make sure features are float32 for GPU speed
# X = X.astype("float32")
# X_test = X_test.astype("float32")
# X_orig = X_orig.astype("float32")

# Convert categorical columns explicitly (LightGBM uses pandas category dtype)
cat_cols = X.select_dtypes(["category"]).columns.tolist()


def objective(trial):
    boosting_type = trial.suggest_categorical("boosting_type", ["gbdt", "dart"])

    param = {
        "objective": "binary",
        "metric": "auc",
        "verbosity": -1,
        "boosting_type": boosting_type,
        # GPU acceleration
        "device": "gpu",
        "gpu_platform_id": 0,
        "gpu_device_id": 0,
        # Learning
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 200, 8000),
        # Tree structure
        "num_leaves": trial.suggest_int("num_leaves", 16, 512),
        "max_depth": trial.suggest_int("max_depth", -1, 128),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 10, 500),
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-3, 10.0, log=True),
        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 5.0),
        # Regularization
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
        # Sampling
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 0, 10),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.5, 1.0),
        # Histogram / tree
        "max_bin": trial.suggest_int("max_bin", 32, 255),
        "grow_policy": trial.suggest_categorical("grow_policy", ["depthwise", "lossguide"]),
        "extra_trees": trial.suggest_categorical("extra_trees", [False, True]),
        # Dart-specific
        "drop_rate": trial.suggest_float("drop_rate", 0.0, 0.5) if boosting_type == "dart" else 0.0,
        "skip_drop": trial.suggest_float("skip_drop", 0.0, 0.5) if boosting_type == "dart" else 0.0,
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    aucs = []

    for train_idx, valid_idx in cv.split(X, y):
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

        model = lgb.LGBMClassifier(**param, verbose=-1)

        model.fit(
            X_train,
            y_train,
            eval_set=[(X_valid, y_valid)],
            eval_metric="auc",
            categorical_feature=cat_cols,  # Pass categorical columns
            callbacks=[lgb.early_stopping(50), optuna.integration.LightGBMPruningCallback(trial, "auc")],  # <-- pruning
        )

        preds = model.predict_proba(X_valid)[:, 1]
        aucs.append(roc_auc_score(y_valid, preds))

    return sum(aucs) / len(aucs)


# Run the tuner
study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner())
study.enqueue_trial(base_params)
study.optimize(objective, n_trials=300)

print("Best params:", study.best_params)
print("Best AUC:", study.best_value)


[I 2025-12-01 23:02:25,247] A new study created in memory with name: no-name-996f97af-92f1-4515-bf09-93c77338f373
[I 2025-12-02 00:36:39,747] Trial 0 finished with value: 0.7285457247893274 and parameters: {'boosting_type': 'dart', 'learning_rate': 0.16722132581464857, 'n_estimators': 2896, 'num_leaves': 366, 'max_depth': 12, 'min_data_in_leaf': 129, 'min_child_weight': 0.2664914177258311, 'min_split_gain': 4.263018771802797, 'lambda_l1': 5.676843225483512e-06, 'lambda_l2': 0.0005589915238473375, 'bagging_fraction': 0.8627042660815395, 'bagging_freq': 6, 'feature_fraction': 0.9820111656156094, 'max_bin': 198, 'grow_policy': 'lossguide', 'extra_trees': False, 'drop_rate': 0.2263886508110925, 'skip_drop': 0.3137245186015346}. Best is trial 0 with value: 0.7285457247893274.


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[2876]	valid_0's auc: 0.710202
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1682]	valid_0's auc: 0.70678
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1738]	valid_0's auc: 0.706502
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[2155]	valid_0's auc: 0.708525
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[3509]	valid_0's auc: 0.71045


[I 2025-12-02 00:38:34,592] Trial 1 finished with value: 0.7084851518319282 and parameters: {'boosting_type': 'gbdt', 'learning_rate': 0.16664726793953055, 'n_estimators': 3512, 'num_leaves': 364, 'max_depth': 3, 'min_data_in_leaf': 349, 'min_child_weight': 5.214630610891436, 'min_split_gain': 4.325236857860066, 'lambda_l1': 0.014263817871190812, 'lambda_l2': 4.2476663576707205e-08, 'bagging_fraction': 0.731418012977123, 'bagging_freq': 1, 'feature_fraction': 0.6014162090993206, 'max_bin': 94, 'grow_policy': 'depthwise', 'extra_trees': True}. Best is trial 0 with value: 0.7285457247893274.


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[53]	valid_0's auc: 0.71122
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[82]	valid_0's auc: 0.709315
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[53]	valid_0's auc: 0.710459
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[45]	valid_0's auc: 0.71061
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[58]	valid_0's auc: 0.711272


[I 2025-12-02 00:38:53,705] Trial 2 finished with value: 0.7105751215663512 and parameters: {'boosting_type': 'gbdt', 'learning_rate': 0.18491994086388344, 'n_estimators': 3360, 'num_leaves': 215, 'max_depth': 102, 'min_data_in_leaf': 123, 'min_child_weight': 0.8377389309513571, 'min_split_gain': 2.1379409028583876, 'lambda_l1': 4.287843726000885e-08, 'lambda_l2': 0.004082375393129097, 'bagging_fraction': 0.5988121614586331, 'bagging_freq': 9, 'feature_fraction': 0.7317286950087334, 'max_bin': 78, 'grow_policy': 'depthwise', 'extra_trees': False}. Best is trial 0 with value: 0.7285457247893274.


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[2374]	valid_0's auc: 0.707848
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[2128]	valid_0's auc: 0.706327
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[3001]	valid_0's auc: 0.708189
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1646]	valid_0's auc: 0.706706
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[3601]	valid_0's auc: 0.710279


[I 2025-12-02 00:42:56,303] Trial 3 finished with value: 0.7078698795549294 and parameters: {'boosting_type': 'gbdt', 'learning_rate': 0.06700804841849019, 'n_estimators': 6412, 'num_leaves': 66, 'max_depth': 6, 'min_data_in_leaf': 20, 'min_child_weight': 0.0046399512062939964, 'min_split_gain': 0.3734030011935707, 'lambda_l1': 1.558504781659084e-08, 'lambda_l2': 0.19947491016236316, 'bagging_fraction': 0.5931532726952309, 'bagging_freq': 1, 'feature_fraction': 0.7506624249120213, 'max_bin': 170, 'grow_policy': 'depthwise', 'extra_trees': True}. Best is trial 0 with value: 0.7285457247893274.
[I 2025-12-02 01:57:05,383] Trial 4 finished with value: 0.7055569662862841 and parameters: {'boosting_type': 'dart', 'learning_rate': 0.0905717572580975, 'n_estimators': 5367, 'num_leaves': 377, 'max_depth': 64, 'min_data_in_leaf': 59, 'min_child_weight': 0.05076656577550214, 'min_split_gain': 2.740856690479723, 'lambda_l1': 5.930579551222987e-07, 'lambda_l2': 4.2590056482813554e-05, 'bagging_fra

In [ ]:
best_params = study.best_params
with open(model_dir + "lgbm_base_params.json", "w") as f:
    json.dump(best_params, f, indent=4)